# 🧠 Managing Memory in LangGraph: Short-Term vs. Long-Term Memory

Welcome to this tutorial on managing **Memory Architectures in LangGraph**!

In LLM application design, **Memory** is divided into two distinct dimensions:
1. **Short-Term Memory (Thread-Scoped Checkpointer)**: Stores state, context, and message history within a single conversation session (`thread_id`). Powered by `InMemorySaver` or persistent checkpointers.
2. **Long-Term Memory (Cross-Thread Store)**: Stores user preferences, profile facts, and history across multiple independent conversation threads (`user_id`). Powered by `InMemoryStore` or persistent stores.

---

### Key Concepts Covered:
- **`InMemorySaver`**: Thread-bound checkpointing for message history.
- **`InMemoryStore`**: Hierarchical key-value store scoped by custom tuple namespaces (e.g., `("users", user_id, "memories")`).
- **`Runtime[Context]`**: Dynamic context injection allowing graph nodes to access runtime variables like `user_id` and the long-term store.
- **Multi-Tenant Memory Isolation**: Scoping long-term memories per user to ensure complete privacy across users.

## 1. Environment Setup & Core Imports

We import essential modules from `langgraph` including graph builders, message state handlers, checkpointers, stores, and runtime context utilities.

In [2]:
# Import standard utility for generating unique identifiers and dataclasses
import uuid
from dataclasses import dataclass

# Import LangGraph graph constructs and MessagesState state schema
from langgraph.graph import (
    StateGraph,
    MessagesState,
    START,
    END
)
# Import Runtime class to access graph execution context and memory store
from langgraph.runtime import Runtime

## 2. Initializing the Chat Model

We instantiate our LLM provider using LangChain's `init_chat_model`. Here, we use Google Gemini (`google_genai:gemini-3.1-flash-lite`) configured with `temperature=0` for consistent and deterministic responses.

In [16]:
from langchain.chat_models import init_chat_model
# Initialize the Google Gemini chat model with temperature=0 for deterministic outputs
model = init_chat_model(
    "google_genai:gemini-3.1-flash-lite",
    temperature=0
)

## 3. Configuring Short-Term Thread Checkpointing

`InMemorySaver` acts as the short-term memory backend. It saves state checkpoints after each node execution, indexed by a `thread_id`.

In [4]:
from langgraph.checkpoint.memory import InMemorySaver
# Initialize in-memory checkpointer for short-term message/state persistence within a single thread
checkpointer = InMemorySaver()

## 4. Configuring Long-Term Cross-Thread Store

`InMemoryStore` acts as a long-term cross-thread memory store. Unlike thread checkpointers, memories saved in `InMemoryStore` persist across different conversation threads for the same user.

In [5]:
from langgraph.store.memory import InMemoryStore
# Initialize in-memory store for long-term user memories persisted across multiple threads
store = InMemoryStore()

## 5. Defining Dynamic Runtime Context Schema

We define a `@dataclass Context` to pass dynamic runtime metadata (such as `user_id`) to the graph nodes upon invocation.

In [6]:
# Define custom runtime context schema to hold dynamic runtime properties (e.g., user_id)
@dataclass
class Context:
    user_id: str

In [7]:
# Assign a sample UUID user ID to test context instantiation
Context.user_id=str(uuid.uuid4())

In [8]:
# Display generated user_id
Context.user_id

'2c8f812d-c495-4361-be01-a3384eece8be'

## 6. Creating the Memory-Aware Assistant Node

The `assistant_node` function accepts `runtime: Runtime[Context]`, enabling it to access both the dynamic `user_id` and the long-term `store`.

### Workflow inside `assistant_node`:
1. **Extract User ID**: `runtime.context.user_id`
2. **Build Namespace**: `("users", user_id, "memories")`
3. **Save Memory**: If the user message starts with `"Remember:"`, extract the fact and write it to `runtime.store.put()`.
4. **Retrieve Memories**: Search `runtime.store` for all memories belonging to the current user.
5. **Inject into System Prompt**: Augment the LLM system prompt with long-term user facts before generating a response.

In [9]:
# Assistant node demonstrating short-term state access & long-term store interaction
def assistant_node(
    state: MessagesState,
    runtime: Runtime[Context]
):
    # Extract user ID from runtime context
    user_id = runtime.context.user_id
    # Construct namespace tuple for multi-tenant memory storage: ("users", user_id, "memories")
    namespace = ("users", user_id, "memories")
    user_message = state["messages"][-1].content
    
    # Check if user message contains a memory update command starting with "remember:"
    if user_message.lower().startswith("remember:"):
        memory = user_message[len("remember:"):].strip()
        # Save memory into long-term store under user's namespace
        runtime.store.put(
            namespace,
            str(uuid.uuid4()),{"data": memory}
        )
        print(f"[MEMORY SAVED]: {memory}")
        
    # Search long-term store for all memories belonging to this user
    memories = runtime.store.search(namespace)
    memory_text = "\n".join(
        memory.value["data"]
        for memory in memories
    )
    if not memory_text:
        memory_text = "No saved memories."
        
    # Build system prompt containing long-term retrieved memories
    system_prompt = f"""
    You are a helpful assistant.

    Long-term memories about this user:

    {memory_text}
    """

    # Invoke LLM with system prompt + conversation history from MessagesState
    response = model.invoke(
        [
            {
                "role": "system",
                "content": system_prompt
            },
            *state["messages"]
        ]
    )
    return {
        "messages": [response]
    }

## 7. Constructing the LangGraph StateGraph

We build the graph using `StateGraph`, passing `MessagesState` as the state schema and `context_schema=Context` so nodes can consume dynamic runtime context.

In [10]:
# Initialize StateGraph passing MessagesState for chat history & Context for runtime user state
builder = StateGraph(
    MessagesState,
    context_schema=Context
)

In [11]:
# Add the memory-aware assistant node to graph builder
builder.add_node(
    "assistant",
    assistant_node
)

In [12]:
# Define execution edges connecting START -> assistant -> END
builder.add_edge(
    START,
    "assistant"
)

builder.add_edge(
    "assistant",
    END
)

## 8. Compiling Graph with Dual Memory Engines

When compiling the state graph, we supply both `checkpointer` (short-term thread memory) and `store` (long-term cross-thread memory).

In [13]:
# Compile graph with both short-term checkpointer and long-term memory store
graph = builder.compile(
    checkpointer=checkpointer,
    store=store
)

## 9. Defining Interactive Chat Runner Function

The `chat` helper function simplifies calling `graph.invoke`:
- **`config`**: Passes `thread_id` to configure short-term state checkpointing.
- **`context`**: Passes `Context(user_id=...)` to supply user context for long-term memory retrieval.

In [14]:
# Helper function to invoke graph with specific user context and thread configuration
def chat(message, user_id, thread_id):

    result = graph.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": message
                }
            ]
        },

        # Thread ID controls short-term checkpointer state
        config={
            "configurable": {
                "thread_id": thread_id
            }
        },

        # Context controls dynamic runtime user_id for long-term store access
        context=Context(user_id=user_id)
    )

    answer = result["messages"][-1].content

    print("\nUSER:")
    print(message)

    print("\nASSISTANT:")
    print(answer)

    print("\n" + "=" * 60)

## 10. Demonstrating Thread Memory & Cross-Thread Persistence

Let's test our agent across multiple turns and threads for user `areeb_123`.

### Turn 1: Short-Term Conversation in `thread_1`

In [17]:
# Turn 1: Provide project info in thread_1 for user areeb_123
chat(
    message="My current project is Agentic RAG.",
    user_id="areeb_123",
    thread_id="thread_1"
)


USER:
My current project is Agentic RAG.

ASSISTANT:
[{'type': 'text', 'text': 'That is a fascinating and rapidly evolving space! **Agentic RAG** (Retrieval-Augmented Generation) is a significant step up from "Naive RAG" because it moves away from a linear pipeline toward a system that can reason, plan, and use tools to find the best possible answer.\n\nTo help you best, I’d love to know where you are in your development process. Are you building from scratch, or are you using frameworks like **LangGraph, LlamaIndex (Workflows/Agents), or CrewAI**?\n\nHere are the core pillars of Agentic RAG that most developers are currently focusing on. Do any of these resonate with your current challenges?\n\n### 1. The "Agentic" Workflow\nUnlike standard RAG, an Agentic system can:\n*   **Self-Correction:** If the initial retrieval yields poor results, the agent decides to re-query with different keywords or search strategies.\n*   **Multi-Step Reasoning:** It breaks down complex user queries into

### Turn 2: Recalling Short-Term Context within `thread_1`
The assistant remembers the current project because both turns occur within `thread_1`.

In [18]:
# Turn 2: Query project in same thread (thread_1) - recalled via Short-Term Checkpointer
chat(
    message="What is my current project?",
    user_id="areeb_123",
    thread_id="thread_1"
)


USER:
What is my current project?

ASSISTANT:
[{'type': 'text', 'text': 'Your current project is **Agentic RAG**.', 'extras': {'signature': 'EnEKbwERTTIP5ASLPnDUT6GNCEDeGJt86fQCNd53lfi/A7CtQRUw0iG5hUGTlu6KSLyAuYPExqER7NB+9xgqsR6F32dwZ4wtKGzt5VCfrDH7AQKFf0ebg8SRIqGo2a/NubpHXxbuuKcZijLZ0EheJ5Fulw=='}}]



### Turn 3: Storing a Long-Term Memory
We send a message starting with `"Remember:"` to trigger long-term memory persistence in `InMemoryStore`.

In [19]:
# Turn 3: Save explicit long-term memory entry via "Remember:" prefix
chat(
    message="Remember: my favorite programming language is Python.",
    user_id="areeb_123",
    thread_id="thread_1"
)

[MEMORY SAVED]: my favorite programming language is Python.

USER:
Remember: my favorite programming language is Python.

ASSISTANT:
[{'type': 'text', 'text': 'Understood! I have updated your profile: your favorite programming language is **Python**, and you are currently working on **Agentic RAG**.\n\nSince you\'re using Python for your Agentic RAG project, are you leaning toward a specific library? Most Python developers in this space are currently using:\n\n*   **LangGraph:** Excellent for building stateful, multi-actor applications with fine-grained control over the agent\'s "thought" loops.\n*   **LlamaIndex:** Very strong for the data-indexing side of RAG and their new "Workflows" feature.\n*   **DSPy:** If you want to move away from manual prompt engineering and toward a more programmatic, "compiler-like" approach to optimizing your RAG pipeline.\n\nLet me know if you need help with any Python-specific implementation details, such as setting up state management in LangGraph or o

### Turn 4: Cross-Thread Memory Sharing in `thread_2`
Now we switch to a **new thread (`thread_2`)**. 
- Short-term conversation history from `thread_1` is **not** present in `thread_2`.
- However, the assistant **still remembers** the favorite programming language because it retrieves long-term facts stored under `user_id="areeb_123"`.

In [20]:
# Turn 4: Switch to thread_2 (new short-term thread) for user areeb_123
# Recalls favorite language via Long-Term InMemoryStore cross-thread sharing
chat(
    message="What is my favorite programming language?",
    user_id="areeb_123",
    thread_id="thread_2"
)


USER:
What is my favorite programming language?

ASSISTANT:
[{'type': 'text', 'text': 'Your favorite programming language is Python.', 'extras': {'signature': 'EnEKbwERTTIPUwpFxyyQjq0CNrc9jzf2jyvv1NjVvkVd60c4PBTyXcUPt7PLzjj9uJhz+kUd2BCoCpqNMSLsIaNDrGXeaZPntruMQ7KXVH2sd6H8KGj2fTLSjhT0eTrWOBdRSLVoK8VHQ4ZhSOqStbHePA=='}}]



### 📊 Architectural Overview: Cross-Thread Long-Term Memory Sharing

```text
                 areeb_123 (User ID)
                     ↓
              InMemoryStore
                     ↓
      "favorite language = Python"
                     ↓
             ┌───────┴───────┐
             ↓               ↓
         thread_1         thread_2
   (Session History 1)  (Session History 2)
```

> **Note**: Both `thread_1` and `thread_2` share the exact same user memory partition in `InMemoryStore` because they belong to `areeb_123`.

In [21]:
# Save memory in explicit thread areeb_thread_1 for user areeb_123
chat(
    message="Remember: my favorite language is Python.",
    user_id="areeb_123",
    thread_id="areeb_thread_1"
)

[MEMORY SAVED]: my favorite language is Python.

USER:
Remember: my favorite language is Python.

ASSISTANT:
[{'type': 'text', 'text': "Got it! I have noted that Python is your favorite programming language. I'll keep that in mind for our future conversations!", 'extras': {'signature': 'EnEKbwERTTIPXCVvC4VPqTDvmS9mQ+e69/ULoBWtf54Diws34TVBt52Ve8f1PqM91aMZmIVN71cGkqoLitWMP8Yk025a5UjHYEyTG3rBB7aqgBN7PSgBQOQQMZ2fXh0wx7Cfs9OM/VK7g3GFivNwkGiAHQ=='}}]



## 11. Multi-Tenant User Memory Isolation

Now let's demonstrate memory isolation between **different users** (`areeb_123` vs `rahul_456`). Memories stored for `rahul_456` will be isolated in their own namespace tuple: `("users", "rahul_456", "memories")`.

### Saving Memory for User `rahul_456` in `rahul_thread_1`

In [22]:
# Save memory for a different user (rahul_456) in rahul_thread_1
chat(
    message="Remember: my favorite language is Java.",
    user_id="rahul_456",
    thread_id="rahul_thread_1"
)

[MEMORY SAVED]: my favorite language is Java.

USER:
Remember: my favorite language is Java.

ASSISTANT:
[{'type': 'text', 'text': "Understood! I have noted that your favorite programming language is Java. I'll keep that in mind for our future conversations!", 'extras': {'signature': 'EnEKbwERTTIPbwh2omT39AMO5eSHTIpV2Du85Vq/Ero1s6YtRBDLdcrk4bhZLYUJy+ay6lLn8GiHa3Hkxx0CRR/S+mBGwFY336TRiOutNvh5CIwplbS/B41waxT+5lOCDWN3DSW0ienwmJKJ2A+TaLAGeQ=='}}]



### Verifying Memory Retrieval for `rahul_456` in `rahul_thread_2`

In [23]:
# Query favorite language for rahul_456 in new thread rahul_thread_2
chat(
    message="What is my favorite programming language?",
    user_id="rahul_456",
    thread_id="rahul_thread_2"
)


USER:
What is my favorite programming language?

ASSISTANT:
[{'type': 'text', 'text': 'Your favorite programming language is Java.', 'extras': {'signature': 'EnEKbwERTTIPG/xM+71zMDyNoFNaRRHPwN3SBqnKiIgtKpqtzGZ0HEUarDUVCNXKWkZ23cmz404zDIMloF7HpA1uEz1jOxD4U87lVk5mtsD5S17pZ4Cr3Pr3hf8iDKQR93i2bRZ7uuZa0tHGgMapYIjHlg=='}}]



### 🗂️ Store Namespace Hierarchy & Data Scoping

```text
users
│
├── areeb_123
│      └── memories
│           └── favorite language = Python
│
└── rahul_456
       └── memories
            └── favorite language = Java
```

> **Key Takeaway**: By keying long-term memories using standard tuple namespaces `("users", user_id, "memories")`, LangGraph guarantees complete data isolation between different users.

### Verifying Isolation: Checking User `areeb_123` in `areeb_thread_2`
We check `areeb_123` again to ensure `rahul_456`'s memory ("Java") did not overwrite or contaminate `areeb_123`'s memory ("Python").

In [24]:
# Query favorite language for areeb_123 in areeb_thread_2 to verify multi-tenant isolation
chat(
    message="What is my favorite programming language?",
    user_id="areeb_123",
    thread_id="areeb_thread_2"
)


USER:
What is my favorite programming language?

ASSISTANT:
[{'type': 'text', 'text': 'Your favorite programming language is Python.', 'extras': {'signature': 'EnEKbwERTTIPsZh5KDmu5d0yHqUQskpWz19AcHH7JRb+R5BpUTyU8rAFcMWI0GsOS+noneMLVVfrcz/rDTheLg5DUrHo0RE5o3floQtTu9+j4YjRmtb9FLIbJ/G+yWvrpyLgDQHyNsGBApQfASHNwxkonw=='}}]



## 12. Direct Inspection of `InMemoryStore`

Finally, we can query and inspect the underlying `InMemoryStore` directly to view all stored JSON memory documents for a specific namespace.

In [25]:
# Target user ID for store query inspection
user_id = "areeb_123" 

In [26]:
# Define memory namespace tuple for store lookup
namespace = (
    "users",
    user_id,
    "memories"
)

### Searching Memory Documents by Namespace

In [27]:
# Search store by namespace and print raw memory dict objects
memories = store.search(namespace)

for memory in memories:
    print(memory.value)

{'data': 'my favorite programming language is Python.'}
{'data': 'my favorite language is Python.'}


### Extracting Specific Memory Data Fields

In [28]:
# Search store directly and extract string values from "data" key
memories = store.search(
    ("users", "areeb_123", "memories")
)

for memory in memories:
    print(memory.value["data"])

my favorite programming language is Python.
my favorite language is Python.
